# Homework 7: Training Without Clean Images
## Name: Jon Knapp
### Date: November 17, 2026
### Class: EE 513 Intro to Image Processing

## AI Disclosure:

AI was used in the following ways:
* Code generation
* Checking my work and my logic
* Checking and assistance with reflections

I used Claude (Anthropic) to write the alignment, training, evaluation and export code and to draft the written answers, which I then reviewed and edited. I checked it by testing the alignment on frames shifted by known sub-pixel amounts before trusting it on the burst, comparing the measured loss gap between the two training methods against the value predicted from the noise variance, repeating training with a second seed to measure how much of any difference is chance, and comparing the ONNX export against PyTorch on the same input. Answers that quote numbers are generated from the notebook's own variables, so they always match the printed output.

**Before running.** Re-download `pyproject.toml` and `uv.lock` (they now include the ONNX packages), then `uv sync --group learning`. Put the sixteen DNGs of your noisiest Project 2 burst in one folder and set `BURST_DIR` below.

**Dry run.** Setting `DEMO_SYNTHETIC = True` runs the whole notebook on a simulated burst: sixteen shifted copies of the green plane of my Homework 2 slanted-edge capture with Poisson-Gaussian noise from my Project 1 noise model at a raised gain. It exists only to test the code before the burst is shot. **No number from a synthetic run goes in the submission.**

**Project 2.** Everything here is Project 2 Part B and Part C on a small network: the burst average is the denoising ground truth, the alignment and the residual-noise estimate are reused as-is, the network is the starting point for the learned stage, and the ONNX export and timing protocol are the ones Part C asks for.

In [ ]:
import os
import platform
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rawpy
import tifffile
import torch
import torch.nn as nn
from IPython.display import Markdown, display
from scipy.ndimage import gaussian_filter

EE513 = Path.cwd().resolve().parent
BURST_DIR = EE513 / "Project 2" / "Project 2 Captures" / "Bursts" / "ISO 1000"   # <- your noisiest burst
DEMO_SYNTHETIC = False      # True only to test the code before the burst exists (see above)

# Saturation measured in Project 1 (Color and Accuracy, item 1): the green sites pile up from about
# 3950 DN, so I use 3940 DN, not the WhiteLevel tag (4095 DN).
SAT_DN = 3940.0

TRAIN_FRACTION = 0.70       # train on the left 70% of the frame, score on the right 30%
PATCH, BATCH, STEPS, LR = 64, 16, 800, 1e-3
SEEDS = (513, 2026)         # first and second training seeds (item 11)
NOISE_MULTIPLE = 4.0        # training noise = 4 x the robust single-frame estimate (item 5)
PEAK = 1.0                  # PSNR peak: saturation, since the data run from 0 (black) to 1 (saturation)
torch.set_num_threads(max(1, os.cpu_count() or 1))
device = torch.device("cpu")    # every timing below is CPU

## Part 1: A clean image from your burst

### Problem 1: Load the sixteen frames of one burst and take the same green sites from each. Use your noisiest burst.
_Deliverable: The number of frames, and the height and width of one half-size image._

In [ ]:
def _lookup(page, name):
    exif = page.tags["ExifTag"].value if "ExifTag" in page.tags else {}
    return exif.get(name, page.tags[name].value if name in page.tags else None)


def capture_info(path):
    """Camera, ISO and exposure from the DNG, for the same-camera, same-ISO check."""
    with tifffile.TiffFile(path) as t:
        p = t.pages[0]
        iso = _lookup(p, "ISOSpeedRatings") or _lookup(p, "PhotographicSensitivity")
        et = _lookup(p, "ExposureTime")
        return {"file": Path(path).name, "lens": _lookup(p, "LensModel"),
                "ISO": int(iso[0] if isinstance(iso, (tuple, list)) else iso),
                "exposure_s": et[0] / et[1] if isinstance(et, (tuple, list)) else float(et)}


def green_sites(path, sat=SAT_DN):
    """One set of green sites of the raw mosaic, black-subtracted and divided by (saturation - black).
    The site is chosen from the file's CFA pattern: the first green in reading order
    (raw[0::2, 1::2] for my BGGR sensor)."""
    with rawpy.imread(str(path)) as r:
        raw = r.raw_image_visible.astype(np.float64)
        pattern, desc = r.raw_pattern.copy(), r.color_desc.decode()
        gi, gj = next((i, j) for i in range(2) for j in range(2) if desc[pattern[i, j]] == "G")
        black = float(r.black_level_per_channel[pattern[gi, gj]])
    return (raw[gi::2, gj::2] - black) / (sat - black), (gi, gj), black


def synthetic_burst(n=16, seed=7):
    """DRY RUN ONLY. A static scene (one real green plane, lightly smoothed), shifted by a random
    sub-pixel amount per frame, with Poisson-Gaussian noise from my Project 1 model at 12x the gain."""
    src = EE513 / "Project 1" / "Homework 2 Image Captures" / "Slanted Edge" / "IMG_0823.DNG"
    scene, _, _ = green_sites(src)
    scene = np.clip(gaussian_filter(scene, 0.7), 0, 1)
    rng = np.random.default_rng(seed)
    K = 0.246 * 12 / (SAT_DN - 528)            # gain in (fraction of full scale) per electron
    read = 1.694 * 12 / (SAT_DN - 528)         # read noise, fraction of full scale
    frames, true_shifts = [], []
    for k in range(n):
        dx, dy = (0.0, 0.0) if k == 0 else rng.normal(0, 0.6, 2)
        moved = cv2.warpAffine(scene, np.float64([[1, 0, dx], [0, 1, dy]]), scene.shape[::-1],
                               flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REFLECT)
        e = rng.poisson(np.clip(moved, 0, None) / K) * K
        frames.append(e + rng.normal(0, read, e.shape))
        true_shifts.append((dx, dy))
    return frames, np.array(true_shifts)


if DEMO_SYNTHETIC:
    display(Markdown("## ⚠ SYNTHETIC DRY RUN: these numbers are not from my camera and are not for submission."))
    frames, TRUE_SHIFTS = synthetic_burst()
    info = pd.DataFrame([{"file": f"synthetic_{k:02d}", "ISO": "sim", "exposure_s": None} for k in range(len(frames))])
    site = (0, 1)
else:
    files = sorted(f for f in BURST_DIR.iterdir() if f.suffix.lower() == ".dng")
    if len(files) < 16:
        raise FileNotFoundError(f"{len(files)} DNGs in {BURST_DIR}; the burst needs 16.")
    info = pd.DataFrame([capture_info(f) for f in files])
    assert info.lens.nunique() == 1 and info.ISO.nunique() == 1 and info.exposure_s.nunique() == 1, \
        "The burst mixes cameras, ISOs or exposure times:\n" + info.to_string()
    loaded = [green_sites(f) for f in files]
    frames, site = [g for g, _, _ in loaded], loaded[0][1]
    TRUE_SHIFTS = None

frames = [f.astype(np.float64) for f in frames]
N = len(frames)
H, W = frames[0].shape
print(f"{N} frames; one half-size green image is {H} x {W} (height x width), green site {site}")
if not DEMO_SYNTHETIC:
    print(f"Camera: {info.lens.iloc[0]}; ISO {info.ISO.iloc[0]}; exposure {info.exposure_s.iloc[0]:.4g} s")

### Problem 2: Align every frame to the first. cv2.phaseCorrelate on a Hann-windowed pair gives the shift to a fraction of a pixel. cv2.warpAffine shifts it back. Report the median and the largest shift.
_Deliverable: Those two numbers, in pixels._

In [ ]:
def dft_friendly(n):
    """Largest even size <= n that OpenCV's DFT uses without padding."""
    m = n
    while cv2.getOptimalDFTSize(m) != m or m % 2:
        m -= 1
    return m


# cv2.phaseCorrelate pads each image to cv2.getOptimalDFTSize. For my 2016-wide green plane that is 2025,
# an odd size, and the sub-pixel peak then comes back half a pixel off: an image correlated with
# ITSELF reports a 0.5 px shift (printed below). So the shift is estimated on a central crop whose
# size needs no padding.
EH, EW = dft_friendly(H), dft_friendly(W)
EST = (slice((H - EH) // 2, (H - EH) // 2 + EH), slice((W - EW) // 2, (W - EW) // 2 + EW))
HANN = cv2.createHanningWindow((EW, EH), cv2.CV_64F)
_full = cv2.phaseCorrelate(frames[0].copy(), frames[0].copy(), cv2.createHanningWindow((W, H), cv2.CV_64F))[0]
_f0 = frames[0].copy()
print(f"Frame 1 against itself, full {W} x {H} (DFT {cv2.getOptimalDFTSize(W)} x {cv2.getOptimalDFTSize(H)}): "
      f"({_full[0]:+.3f}, {_full[1]:+.3f}) px; on the {EW} x {EH} crop: "
      f"{tuple(round(v, 3) for v in cv2.phaseCorrelate(frames[0][EST].copy(), frames[0][EST].copy(), HANN)[0])} px")


def estimate_shift(ref, img, method="opencv"):
    """(dx, dy) by which `img`'s content is displaced relative to `ref`, in pixels, measured on the
    DFT-friendly central crop. 'opencv' is cv2.phaseCorrelate on the Hann-windowed pair, as the
    assignment specifies; 'skimage' is scikit-image's upsampled phase correlation (1/100 px grid),
    kept as an independent cross-check."""
    # .copy() matters: when no padding is needed, cv2.phaseCorrelate multiplies its inputs by the
    # window IN PLACE, which would silently darken frame 1 (and any array these slices view) each call.
    a, b = ref[EST].copy(), img[EST].copy()
    if method == "opencv":
        (dx, dy), _ = cv2.phaseCorrelate(a, b, HANN)
        return float(dx), float(dy)
    from skimage.registration import phase_cross_correlation
    s, _, _ = phase_cross_correlation(a * HANN, b * HANN, upsample_factor=100)
    return float(-s[1]), float(-s[0])


def shift_image(img, dx, dy):
    """Move the content by (dx, dy) with cubic interpolation."""
    return cv2.warpAffine(img, np.float64([[1, 0, dx], [0, 1, dy]]), (img.shape[1], img.shape[0]),
                          flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REFLECT)


# Known-answer test first: displace frame 0 by known sub-pixel amounts and see what comes back.
known = [(0.0, 0.0), (0.25, -0.10), (0.40, 0.70), (-0.60, 0.30), (1.30, -1.70)]
rows = []
for dx, dy in known:
    moved = shift_image(frames[0], dx, dy) + np.random.default_rng(0).normal(0, frames[0].std() * 0.02, frames[0].shape)
    for m in ("opencv", "skimage"):
        ex, ey = estimate_shift(frames[0], moved, m)
        rows.append({"true (dx, dy)": f"({dx:+.2f}, {dy:+.2f})", "method": m, "error dx": ex - dx, "error dy": ey - dy})
selftest = pd.DataFrame(rows)
selftest["|error|"] = np.hypot(selftest["error dx"], selftest["error dy"])
display(selftest.pivot(index="true (dx, dy)", columns="method", values="|error|").style.format("{:.3f}")
        .set_caption("Known-shift test on frame 0: |estimated - true| shift, px"))

In [ ]:
ALIGN_METHOD = "opencv"      # the assignment's method; switch to "skimage" if the test above shows it is more accurate

shifts = {m: np.array([estimate_shift(frames[0], f, m) for f in frames]) for m in ("opencv", "skimage")}
assert np.array_equal(frames[0], _f0), "a frame was modified in place during shift estimation"
aligned = [shift_image(f, -dx, -dy) for f, (dx, dy) in zip(frames, shifts[ALIGN_METHOD])]
mag = np.hypot(*shifts[ALIGN_METHOD][1:].T)                   # frames 2..N relative to frame 1
MARGIN = int(np.ceil(np.abs(shifts[ALIGN_METHOD]).max())) + 4  # crop the edge the warp filled in

shift_table = pd.DataFrame(shifts[ALIGN_METHOD], columns=["dx (px)", "dy (px)"])
shift_table["|shift| (px)"] = np.hypot(shift_table["dx (px)"], shift_table["dy (px)"])
shift_table["skimage |shift| (px)"] = np.hypot(*shifts["skimage"].T)
if TRUE_SHIFTS is not None:
    shift_table["true |shift| (synthetic)"] = np.hypot(*TRUE_SHIFTS.T)
display(shift_table.style.format("{:.3f}").set_caption(f"Shift of each frame relative to frame 1 ({ALIGN_METHOD})"))
print(f"Shift relative to the first frame, frames 2-{N} ({ALIGN_METHOD}): "
      f"median {np.median(mag):.3f} px, largest {mag.max():.3f} px; "
      f"the two methods differ by up to {np.abs(shifts['opencv'] - shifts['skimage']).max():.3f} px")

### Problem 3: Average the aligned frames. That's your clean image. Show one frame and the average side by side, same crop, full scale, somewhere with fine detail.
_Deliverable: One figure._

In [ ]:
clean = np.mean(aligned, axis=0)
inner = (slice(MARGIN, H - MARGIN), slice(MARGIN, W - MARGIN))   # every later step works inside this
clean, aligned = clean[inner], [a[inner] for a in aligned]
H, W = clean.shape
SPLIT = int(TRAIN_FRACTION * W)                                    # columns < SPLIT train, >= SPLIT score

# Crop with the most fine detail: highest local variance of the average after removing slow shading.
detail = gaussian_filter((clean - gaussian_filter(clean, 4)) ** 2, 24)
cy, cx = np.unravel_index(np.argmax(detail[64:-64, 64:-64]), (H - 128, W - 128))
cy, cx = cy + 64, cx + 64
C3 = (slice(cy - 64, cy + 64), slice(cx - 64, cx + 64))

lo, hi = np.percentile(clean[C3], [0.5, 99.5])
fig, axes = plt.subplots(1, 2, figsize=(12, 6.2))
for ax, im, t in zip(axes, (aligned[0][C3], clean[C3]), ("one frame (frame 1)", f"average of {N} aligned frames")):
    ax.imshow(im, cmap="gray", vmin=lo, vmax=hi, interpolation="nearest")
    ax.set_title(t); ax.axis("off")
fig.suptitle(f"128 x 128 green-site pixels at row {cy}, column {cx}, shown 1:1 in pixels, same linear gray scale",
             fontsize=10)
fig.tight_layout(); fig.savefig("hw7_item3_frame_vs_average.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"Cropped {MARGIN} px from every side after alignment: clean image {H} x {W}; train columns 0-{SPLIT - 1}, "
      f"held-out columns {SPLIT}-{W - 1}")

### Problem 4: Estimate the noise in one frame from the difference of two aligned frames, divided by the square root of two. Do it twice. Once as a standard deviation, and once as the median absolute deviation divided by 0.6745.
_Deliverable: The two numbers, and one sentence on what their disagreement tells you about your alignment._

In [ ]:
def noise_estimates(a, b):
    d = (a - b) / np.sqrt(2)
    sd = d.std(ddof=1)
    robust = np.median(np.abs(d - np.median(d))) / 0.6745
    return sd, robust, d


SIGMA_SD, SIGMA_ROBUST, diff01 = noise_estimates(aligned[0], aligned[1])
# The same pair aligned with the other method, to see whether the disagreement is about alignment.
other = "skimage" if ALIGN_METHOD == "opencv" else "opencv"
f1_other = shift_image(frames[1], -shifts[other][1][0], -shifts[other][1][1])[inner]
sd_o, rob_o, _ = noise_estimates(aligned[0], f1_other)
# And between two frames whose estimated shift is smallest, where interpolation matters least.
k_small = 1 + int(np.argmin(mag))
sd_s, rob_s, _ = noise_estimates(aligned[0], aligned[k_small])

noise_tab = pd.DataFrame({
    "std / sqrt(2)": [SIGMA_SD, sd_o, sd_s], "MAD / 0.6745 / sqrt(2)": [SIGMA_ROBUST, rob_o, rob_s]},
    index=[f"frames 1 and 2, {ALIGN_METHOD} (used)", f"frames 1 and 2, {other}", f"frames 1 and {k_small + 1} (smallest shift)"])
noise_tab["ratio"] = noise_tab.iloc[:, 0] / noise_tab.iloc[:, 1]
display(noise_tab.style.format("{:.5f}", subset=noise_tab.columns[:2]).format("{:.2f}", subset=["ratio"])
        .set_caption("Single-frame noise, fraction of full scale (x 3412 for DN)"))

# Where are the large differences? If they sit on edges, the frames are not quite registered.
edges = np.hypot(*np.gradient(gaussian_filter(clean, 1)))
tail = np.abs(diff01) > 4 * SIGMA_ROBUST
on_edges = edges[tail].mean() / edges.mean() if tail.any() else float("nan")
print(f"{tail.mean():.3%} of pixels lie beyond 4 robust sigma (Gaussian: 0.006%); their mean gradient is "
      f"{on_edges:.1f}x the frame average")

In [ ]:
ratio = SIGMA_SD / SIGMA_ROBUST
verdict = ("they agree, so the difference image is close to Gaussian and the frames are registered to well under the "
           "scale of the edges" if ratio < 1.2 else
           f"the standard deviation is {ratio:.1f}x the robust estimate because the difference image has heavy tails: "
           f"pixels beyond 4σ sit on edges (mean gradient {on_edges:.1f}x the average), so the frames are not "
           f"perfectly registered and edge residuals inflate the standard deviation while the median ignores them")
display(Markdown(
    f"Standard deviation **{SIGMA_SD:.5f}**, robust (MAD / 0.6745) **{SIGMA_ROBUST:.5f}** of full scale "
    f"({SIGMA_SD * (SAT_DN - 528):.1f} and {SIGMA_ROBUST * (SAT_DN - 528):.1f} DN). The ratio is {ratio:.2f}: {verdict}. "
    f"Interpolating a frame during alignment also smooths its noise slightly, which lowers both estimates; the frame pair "
    f"with the smallest shift gives {rob_s:.5f}. The robust value is the one I use for the noise level, and the residual "
    f"noise left in the {N}-frame average is about {SIGMA_ROBUST / np.sqrt(N):.5f}, {np.sqrt(N):.0f}x below one frame."))

### Problem 5: Build your training pair. Add two independent draws of Gaussian noise to your clean image, both at four times the robust estimate from item 4. Report the PSNR of one noisy copy against the clean image, and say what peak you used.
_Deliverable: The PSNR and the peak._

In [ ]:
def psnr(est, ref, peak=PEAK):
    mse = float(np.mean((np.asarray(est, np.float64) - np.asarray(ref, np.float64)) ** 2))
    return 10 * np.log10(peak ** 2 / mse)


SIGMA_TRAIN = NOISE_MULTIPLE * SIGMA_ROBUST
rng_pair = np.random.default_rng(12345)
noisy1 = clean + rng_pair.normal(0, SIGMA_TRAIN, clean.shape)
noisy2 = clean + rng_pair.normal(0, SIGMA_TRAIN, clean.shape)
PSNR_NOISY = psnr(noisy1, clean)
display(Markdown(
    f"Training noise σ = {NOISE_MULTIPLE:g} × {SIGMA_ROBUST:.5f} = **{SIGMA_TRAIN:.5f}** of full scale. "
    f"PSNR of the first noisy copy against the clean image: **{PSNR_NOISY:.2f} dB**, peak = **{PEAK:g}**, "
    f"which is saturation ({SAT_DN:.0f} DN minus black), because the data were divided by that span. "
    f"No clipping, so the noise stays zero-mean. As a check, a Gaussian of that σ predicts "
    f"{10 * np.log10(PEAK ** 2 / SIGMA_TRAIN ** 2):.2f} dB."))

## Part 2: Two ways to train

### Problem 6: Define the network and report how many parameters it has.
_Deliverable: The parameter count._

In [ ]:
class DenoiseStack(nn.Module):
    """The lecture 11 stack: six 3x3 convolutions, 32 channels, ReLU between them. It predicts the
    noise, so the output is the input minus what the network produced."""

    def __init__(self, width=32, depth=6):
        super().__init__()
        layers = [nn.Conv2d(1, width, 3, padding=1), nn.ReLU()]
        for _ in range(depth - 2):
            layers += [nn.Conv2d(width, width, 3, padding=1), nn.ReLU()]
        layers.append(nn.Conv2d(width, 1, 3, padding=1))
        self.body = nn.Sequential(*layers)

    def forward(self, noisy):
        return noisy - self.body(noisy)


n_params = sum(p.numel() for p in DenoiseStack().parameters())
by_hand = (1 * 32 * 9 + 32) + 4 * (32 * 32 * 9 + 32) + (32 * 1 * 9 + 1)
assert n_params == by_hand
print(f"Parameters: {n_params:,}  (by hand: 320 + 4 x 9,248 + 289 = {by_hand:,})")

In [ ]:
def train(inp, target, seed, steps=STEPS, batch=BATCH, patch=PATCH, lr=LR):
    """Random patch x patch crops from the training columns only. Same seed -> same initial
    weights and same crops, so two runs that differ only in the target are directly comparable."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    net = DenoiseStack().to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    X = torch.from_numpy(inp[:, :SPLIT].astype(np.float32))
    Y = torch.from_numpy(target[:, :SPLIT].astype(np.float32))
    losses, t0 = [], time.perf_counter()
    for _ in range(steps):
        ys = rng.integers(0, X.shape[0] - patch + 1, batch)
        xs = rng.integers(0, X.shape[1] - patch + 1, batch)
        xb = torch.stack([X[y:y + patch, x:x + patch] for y, x in zip(ys, xs)])[:, None]
        yb = torch.stack([Y[y:y + patch, x:x + patch] for y, x in zip(ys, xs)])[:, None]
        opt.zero_grad()
        loss = loss_fn(net(xb), yb)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    net.eval()
    return net, np.array(losses), time.perf_counter() - t0


def run(net, img):
    with torch.no_grad():
        return net(torch.from_numpy(img.astype(np.float32))[None, None])[0, 0].numpy().astype(np.float64)


def smooth(x, n=50):
    return np.convolve(x, np.ones(n) / n, mode="valid")


def plot_loss(losses, title, fname, extra=None):
    fig, ax = plt.subplots(figsize=(7.5, 4))
    ax.semilogy(losses, color="0.75", lw=0.8, label="per step")
    ax.semilogy(np.arange(49, len(losses)), smooth(losses), lw=1.8, label="50-step mean")
    for y, lab, ls in (extra or []):
        ax.axhline(y, ls=ls, color="k", lw=1, label=lab)
    ax.set_xlabel(f"step (batch {BATCH} of {PATCH} x {PATCH} patches, Adam lr {LR:g})"); ax.set_ylabel("training MSE")
    ax.set_title(title); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(fname, dpi=150); plt.show()

### Problem 7: Train it the ordinary way. Input the first noisy copy, target the clean image. Plot the loss against step.
_Deliverable: One loss curve._

In [ ]:
net_sup, loss_sup, t_sup = train(noisy1, clean, SEEDS[0])
print(f"Supervised training: {STEPS} steps in {t_sup:.0f} s")
plot_loss(loss_sup, "Supervised: noisy copy 1 -> clean", "hw7_item7_supervised_loss.png",
          [(SIGMA_TRAIN ** 2, f"input noise variance σ² = {SIGMA_TRAIN ** 2:.2e}", "--")])

### Problem 8: Train a second copy the Noise2Noise way. Same input, but target the second noisy copy. Change nothing else. Plot the loss against step.
_Deliverable: One loss curve._

In [ ]:
net_n2n, loss_n2n, t_n2n = train(noisy1, noisy2, SEEDS[0])
print(f"Noise2Noise training: {STEPS} steps in {t_n2n:.0f} s")
plot_loss(loss_n2n, "Noise2Noise: noisy copy 1 -> noisy copy 2", "hw7_item8_n2n_loss.png",
          [(SIGMA_TRAIN ** 2, f"σ² = {SIGMA_TRAIN ** 2:.2e} (the floor this loss can't go below)", "--")])

### Problem 9: The two losses settle at different levels. Say why. Give the number the gap should equal, then check it.
_Deliverable: One sentence, the predicted number, and the measured gap._

In [ ]:
LAST = 200
gap = loss_n2n[-LAST:].mean() - loss_sup[-LAST:].mean()
gap_se = np.hypot(loss_n2n[-LAST:].std(ddof=1), loss_sup[-LAST:].std(ddof=1)) / np.sqrt(LAST)
display(Markdown(
    f"The Noise2Noise target is the clean image plus a second, independent noise draw that the input carries no "
    f"information about, so no network can predict it and its whole variance, σ², stays in the loss on top of "
    f"whatever error the network makes against the clean image. "
    f"**Predicted gap: σ² = {SIGMA_TRAIN ** 2:.3e}.** "
    f"**Measured gap** (mean of the last {LAST} steps): {gap:.3e} ± {gap_se:.1e} (standard error, treating steps as "
    f"independent), {gap / SIGMA_TRAIN ** 2:.2f} times the prediction."))

### Problem 10: Report PSNR against the clean image on the held-out right 30 percent, for the noisy input and for each of the two networks.
_Deliverable: Three PSNR values._

In [ ]:
HELD = (slice(None), slice(SPLIT, None))
held_clean, held_noisy = clean[HELD], noisy1[HELD]
out_sup, out_n2n = run(net_sup, held_noisy), run(net_n2n, held_noisy)
P10 = {"noisy input": psnr(held_noisy, held_clean),
       "supervised (clean target)": psnr(out_sup, held_clean),
       "Noise2Noise (noisy target)": psnr(out_n2n, held_clean)}
display(pd.Series(P10, name="PSNR (dB)").to_frame().style.format("{:.2f}").set_caption(
    f"Held-out right {100 - 100 * TRAIN_FRACTION:.0f}% ({held_clean.shape[1]} x {held_clean.shape[0]} px), "
    f"against the burst average; peak {PEAK:g} = saturation"))

dh = detail[:, SPLIT:]                                  # most detailed 128 x 128 crop inside the held-out region
y10, x10 = np.unravel_index(np.argmax(dh[64:-64, 64:-64]), (dh.shape[0] - 128, dh.shape[1] - 128))
C10 = (slice(y10, y10 + 128), slice(x10, x10 + 128))
lo, hi = np.percentile(held_clean[C10], [0.5, 99.5])
fig, axes = plt.subplots(1, 4, figsize=(16, 4.3))
for ax, im, t in zip(axes, (held_clean, held_noisy, out_sup, out_n2n),
                     ("clean (average)", *[f"{k}\n{v:.2f} dB" for k, v in P10.items()])):
    ax.imshow(im[C10], cmap="gray", vmin=lo, vmax=hi, interpolation="nearest"); ax.set_title(t, fontsize=9); ax.axis("off")
fig.suptitle("Held-out region, most detailed 128 x 128 crop, 1:1, display range set by the clean image", fontsize=10, y=1.04)
fig.tight_layout(); fig.savefig("hw7_item10_heldout.png", dpi=150, bbox_inches="tight"); plt.show()

## Part 3: Checking the number

### Problem 11: Train the Noise2Noise network again from a different random seed. Change nothing else. Score it the same way, then compare the gap between the two seeds with the gap you found in item 10.
_Deliverable: The second score, and one sentence on which difference is larger and what that lets you claim._

In [ ]:
net_n2n_b, loss_n2n_b, _ = train(noisy1, noisy2, SEEDS[1])
P11 = psnr(run(net_n2n_b, held_noisy), held_clean)
seed_gap = abs(P11 - P10["Noise2Noise (noisy target)"])
method_gap = abs(P10["supervised (clean target)"] - P10["Noise2Noise (noisy target)"])
claim = (f"the seed-to-seed difference ({seed_gap:.2f} dB) is larger than the supervised-versus-Noise2Noise difference "
         f"({method_gap:.2f} dB), so the two methods are indistinguishable at this level: I can claim Noise2Noise matched "
         f"the supervised network, not that either one won"
         if seed_gap >= method_gap else
         f"the method difference ({method_gap:.2f} dB) is larger than the seed-to-seed difference ({seed_gap:.2f} dB), but "
         f"with one seed of each that is {method_gap / max(seed_gap, 1e-9):.1f}x a single draw of the spread, so I can claim "
         f"a difference only tentatively; several seeds per method would put an uncertainty on it")
display(Markdown(
    f"Noise2Noise, seed {SEEDS[1]}: **{P11:.2f} dB** (seed {SEEDS[0]}: {P10['Noise2Noise (noisy target)']:.2f} dB). "
    f"Comparing the gaps, {claim}."))

### Problem 12: Score the Noise2Noise network on eight random 128 by 128 crops of the held-out region. Then score it on five fresh noise draws of one fixed crop. Report a mean and a standard deviation for each.
_Deliverable: Two mean-and-spread values._

In [ ]:
CROP = 128
rng_c = np.random.default_rng(8)
hh, hw = held_clean.shape
corners = list(zip(rng_c.integers(0, hh - CROP + 1, 8), rng_c.integers(0, hw - CROP + 1, 8)))

crop_rows = []
for y, x in corners:
    sl = (slice(y, y + CROP), slice(x, x + CROP))
    crop_rows.append({"crop (row, col)": f"({y}, {x + SPLIT})", "input": psnr(held_noisy[sl], held_clean[sl]),
                      "Noise2Noise": psnr(out_n2n[sl], held_clean[sl])})
crops = pd.DataFrame(crop_rows)

y0, x0 = corners[0]                                      # the fixed crop: the first of the eight
fixed = (slice(y0, y0 + CROP), slice(x0, x0 + CROP))
draw_rows = []
for k in range(5):
    nk = held_clean + np.random.default_rng(9000 + k).normal(0, SIGMA_TRAIN, held_clean.shape)
    ok = run(net_n2n, nk)                                # whole held-out region, so the crop sees real context
    draw_rows.append({"draw": k + 1, "input": psnr(nk[fixed], held_clean[fixed]), "Noise2Noise": psnr(ok[fixed], held_clean[fixed])})
draws = pd.DataFrame(draw_rows)

summary12 = pd.DataFrame({
    "N2N mean (dB)": [crops["Noise2Noise"].mean(), draws["Noise2Noise"].mean()],
    "N2N s.d. (dB)": [crops["Noise2Noise"].std(ddof=1), draws["Noise2Noise"].std(ddof=1)],
    "input mean (dB)": [crops["input"].mean(), draws["input"].mean()],
    "gain s.d. (dB)": [(crops["Noise2Noise"] - crops["input"]).std(ddof=1), (draws["Noise2Noise"] - draws["input"]).std(ddof=1)]},
    index=["8 random crops, one noise draw", "1 fixed crop, 5 fresh noise draws"])
display(summary12.style.format("{:.2f}"))
display(crops.style.format(precision=2).hide(axis="index"))

### Problem 13: One of those spreads is much larger. Say which, and what that means if you report a single PSNR from a single crop.
_Deliverable: Two sentences._

In [ ]:
s_crop, s_draw = summary12["N2N s.d. (dB)"]
r13 = max(s_crop, s_draw) / max(min(s_crop, s_draw), 1e-9)
if s_crop > s_draw:
    first = (f"The **crop-to-crop** spread is the larger ({s_crop:.2f} dB across crops against {s_draw:.2f} dB across "
             f"noise draws, {r13:.1f}x): PSNR depends more on what is in the crop (flat regions score high, edges and "
             f"texture low) than on the particular noise draw.")
else:
    first = (f"Here the **draw-to-draw** spread is the larger ({s_draw:.2f} dB against {s_crop:.2f} dB across crops, "
             f"{r13:.1f}x), so this held-out region is unusually uniform in content.")
display(Markdown(
    first + f" A single PSNR from a single crop therefore carries an uncertainty of about ±{max(s_crop, s_draw):.1f} dB "
    f"that the number alone doesn't show, so a comparison between methods has to use the same crops (paired), "
    f"average over many of them, and state how they were chosen."))

### Problem 14: Make a correlated noise field. Smooth Gaussian noise with a 1-pixel Gaussian, then rescale it to the standard deviation you trained at. Add it to the clean image and run the same network. Report the PSNR going in and coming out.
_Deliverable: Two PSNR values._

In [ ]:
corr = gaussian_filter(np.random.default_rng(14).normal(0, 1, clean.shape), sigma=1.0)
corr *= SIGMA_TRAIN / corr.std()
noisy_corr = (clean + corr)[HELD]
P14_in, P14_out = psnr(noisy_corr, held_clean), psnr(run(net_n2n, noisy_corr), held_clean)
lag1 = np.corrcoef(corr[:, :-1].ravel(), corr[:, 1:].ravel())[0, 1]
display(Markdown(
    f"Correlated noise at the same σ ({corr.std():.5f}; neighbor correlation {lag1:.2f}, against 0 for the training "
    f"noise): **{P14_in:.2f} dB in, {P14_out:.2f} dB out** ({P14_out - P14_in:+.2f} dB). On white noise at the same σ the "
    f"same network gained {P10['Noise2Noise (noisy target)'] - P10['noisy input']:+.2f} dB: it learned what white noise "
    f"looks like, and smooth blobs of the same variance look to it like image structure, so it leaves most of them in."))

### Problem 15: Run the network on one real raw frame from your burst. Score it against the same clean image. It'll do much less than item 10 suggests.
_Deliverable: The PSNR in and out, and two sentences on the two separate reasons it does worse._

In [ ]:
REAL = 1                                               # frame 2 of the burst (aligned); frame 1 is the alignment reference
real = aligned[REAL][HELD]
P15_in, P15_out = psnr(real, held_clean), psnr(run(net_n2n, real), held_clean)

# Separate the reasons. (a) Level: white Gaussian noise at the real frame's sigma, through the same network.
white_real = held_clean + np.random.default_rng(15).normal(0, SIGMA_ROBUST, held_clean.shape)
Pw_in, Pw_out = psnr(white_real, held_clean), psnr(run(net_n2n, white_real), held_clean)
# (b) Structure: the real frame's noise against the white-noise assumption.
resid = real - held_clean
real_lag1 = np.corrcoef(resid[:, :-1].ravel(), resid[:, 1:].ravel())[0, 1]
q = np.quantile(held_clean, [0.25, 0.75])
sd_dark, sd_bright = resid[held_clean < q[0]].std(), resid[held_clean > q[1]].std()
# The reference contains 1/N of this frame's noise, and residual noise sigma/sqrt(N) of its own.
ceiling = 10 * np.log10(PEAK ** 2 / (SIGMA_ROBUST ** 2 / N))

display(pd.DataFrame({"PSNR in (dB)": [P15_in, Pw_in, P10["noisy input"]],
                      "PSNR out (dB)": [P15_out, Pw_out, P10["Noise2Noise (noisy target)"]]},
                     index=["real frame", "white Gaussian at the real frame's σ", "item 10: white Gaussian at 4σ (trained)"])
        .assign(**{"change (dB)": lambda d: d.iloc[:, 1] - d.iloc[:, 0]}).style.format("{:.2f}"))
print(f"Real-frame noise: neighbor correlation {real_lag1:.2f}; s.d. {sd_dark:.5f} in the darkest quarter vs "
      f"{sd_bright:.5f} in the brightest ({sd_bright / sd_dark:.2f}x). Measurable ceiling against this reference ≈ {ceiling:.1f} dB")

In [ ]:
struct_cost = (Pw_out - Pw_in) - (P15_out - P15_in)          # extra loss from the noise's structure alone
display(Markdown(
    f"Real frame: **{P15_in:.2f} dB in, {P15_out:.2f} dB out** ({P15_out - P15_in:+.2f} dB, against "
    f"{P10['Noise2Noise (noisy target)'] - P10['noisy input']:+.2f} dB in item 10). "
    f"First, the level is wrong: the network was trained at {NOISE_MULTIPLE:g}× the real noise, and even on white Gaussian "
    f"noise at the real level it gains only {Pw_out - Pw_in:+.2f} dB, because there is only a sixteenth of the noise power "
    f"to remove, while a network tuned to noise four times stronger also smooths away real detail, which now weighs "
    f"relatively more. Second, the noise is the wrong kind: the real residual has neighbor correlation "
    f"{real_lag1:.2f} (sensor processing and the alignment's interpolation) and grows with signal "
    f"({sd_bright / sd_dark:.1f}x from the darkest to the brightest quarter, shot noise), where the training noise was white "
    f"and uniform; against white noise at the same level that costs another {struct_cost:+.2f} dB here, and item 14 showed "
    f"how much stronger correlation hurts. (The reference also holds 1/{N} of this frame's noise and its own residual "
    f"σ/√{N}, which caps what a perfect denoiser could score against it near {ceiling:.0f} dB.)"))

## Part 4: ONNX and timing

### Problem 16: Export the Noise2Noise network to ONNX and load it with onnxruntime. Run the same input through it and through PyTorch. Report the largest absolute difference.
_Deliverable: That difference, and the number you think it should be compared against._

In [ ]:
import onnxruntime as ort

ONNX_PATH = Path("hw7_noise2noise.onnx")
net_n2n.eval()
example = torch.from_numpy(noisy1[:PATCH, :PATCH].astype(np.float32))[None, None]
dyn = torch.export.Dim.DYNAMIC
torch.onnx.export(net_n2n, (example,), str(ONNX_PATH), dynamo=True,
                  input_names=["noisy"], output_names=["denoised"],
                  dynamic_shapes={"noisy": {2: dyn, 3: dyn}})       # height and width free
sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
print(f"Exported {ONNX_PATH} ({ONNX_PATH.stat().st_size / 1024:.0f} kB); input '{inp.name}' shape {inp.shape} "
      f"(symbolic height and width); onnxruntime {ort.__version__}")

x_test = held_noisy.astype(np.float32)[None, None]                  # a different size from the export example
onnx_out = sess.run(None, {inp.name: x_test})[0][0, 0]
torch_out = run(net_n2n, held_noisy).astype(np.float32)
d16 = np.abs(onnx_out - torch_out)
eps32 = np.finfo(np.float32).eps
scale = np.abs(torch_out).max()
display(Markdown(
    f"On the {x_test.shape[3]} × {x_test.shape[2]} held-out region (not the {PATCH} × {PATCH} export size): largest "
    f"|ONNX − PyTorch| = **{d16.max():.2e}**. The comparison is float32 rounding at the size of the output: eps × "
    f"max|output| = {eps32:.2e} × {scale:.3f} = {eps32 * scale:.2e}, and each output sums about "
    f"{9 * 32 * 5 + 9:,} products through six layers in a different order, so a few to a few tens of eps is "
    f"agreement; {d16.max() / (eps32 * scale):.1f} eps here. A wrong export (a missing layer, a transposed kernel) would "
    f"differ by the size of the noise, about {SIGMA_TRAIN:.3f}."))

### Problem 17: Time one forward pass at 512 by 512 through onnxruntime. Ten runs, after one warm-up run you throw away. Report the mean and the standard deviation. Say what hardware and what image size they belong to.
_Deliverable: The timing with its spread, the size, and the hardware._

In [ ]:
def cpu_name():
    try:
        if platform.system() == "Windows":
            import winreg
            k = winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, r"HARDWARE\DESCRIPTION\System\CentralProcessor\0")
            return winreg.QueryValueEx(k, "ProcessorNameString")[0].strip()
        if platform.system() == "Linux":
            for line in open("/proc/cpuinfo"):
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except Exception:
        pass
    return platform.processor() or platform.machine()


S = 512
x512 = held_noisy[:S, :S].astype(np.float32)[None, None]
assert x512.shape[2:] == (S, S), f"held-out region is smaller than {S} x {S}: {held_noisy.shape}"


def time_it(fn, runs=10):
    fn()                                                   # warm-up, discarded
    t = []
    for _ in range(runs):
        t0 = time.perf_counter(); fn(); t.append(time.perf_counter() - t0)
    return 1e3 * np.array(t)


t_onnx = time_it(lambda: sess.run(None, {inp.name: x512}))
xt512 = torch.from_numpy(x512)
with torch.no_grad():
    t_torch = time_it(lambda: net_n2n(xt512))
HW = (f"{cpu_name()}, {os.cpu_count()} logical cores, {platform.system()} {platform.release()}; "
      f"onnxruntime {ort.__version__} CPUExecutionProvider, torch {torch.__version__}")
display(Markdown(
    f"**{t_onnx.mean():.1f} ± {t_onnx.std(ddof=1):.1f} ms** per forward pass (mean ± s.d. of 10 runs after one discarded "
    f"warm-up) through onnxruntime, for one **{S} × {S} single-channel** float32 image on the CPU: {HW}. For comparison, "
    f"PyTorch on the same CPU took {t_torch.mean():.1f} ± {t_torch.std(ddof=1):.1f} ms. Individual runs: "
    f"{', '.join(f'{v:.1f}' for v in t_onnx)} ms."))

## Part 5: Reflections
### Problem 18: What did you learn from this assignment?
_Deliverable: A brief written response._

- **A denoiser can be trained without a clean image**

  The Noise2Noise target is noisy, but its noise is independent of the input and has zero mean, so the network that minimizes the loss is the same one that would minimize it against the clean image. The only visible difference is the loss settling σ² higher, which I could predict before training and then measure.

- **The burst average is a real ground truth, with a noise floor I can put a number on**

  Sixteen aligned frames cut the noise by four. The alignment matters: comparing the standard deviation and the robust MAD estimate of the frame difference shows whether the frames are actually registered, and testing the shift estimator on known sub-pixel shifts showed how accurate it is before I trusted it.

- **A PSNR needs its spread before it means anything**

  Crop-to-crop variation was much larger than noise-draw variation, and retraining with another seed moved the score by an amount I had to compare against the method difference before claiming one method beat the other.

- **A learned denoiser is only as general as its training noise**

  Correlated noise at the same variance, or the real frame's weaker, signal-dependent noise, got much less benefit than the white Gaussian noise it was trained on. For Project 2 the training noise has to come from my measured Poisson-Gaussian model, at the levels I'll actually test.

- **ONNX export is checkable against PyTorch to float32 rounding**

  Exporting with free height and width, running a differently sized input, and comparing to PyTorch is a quick end-to-end check of the deployed model, and the timing protocol (warm-up, ten runs, spread, stated hardware) is what Part C needs.

### Problem 19: What questions do you have for me?
_Deliverable: A brief written response._

For Project 2 Part B, the assignment says to train on synthetic Poisson-Gaussian noise and evaluate on real burst pairs. Since real noise is spatially correlated after alignment (item 15), should the evaluation input be an unwarped frame (with the average warped to it instead), so the test noise isn't changed by interpolation?

### Problem 20: If you used AI tools, say what you used them for and how you verified the output.
_Deliverable: Two sentences, or a statement that you didn't use them._

I used Claude (Anthropic) to write the alignment, training, scoring and ONNX code and to draft these answers, which I reviewed and edited. I verified it with known-answer tests: the shift estimator against known sub-pixel shifts, the parameter count by hand, the Noise2Noise loss gap against its predicted value σ², the seed-to-seed spread against the method difference, and the ONNX output against PyTorch at a size different from the export. The answers that quote numbers are generated from the notebook's own variables.